# From multipliers back to a prediction

Solving the dual gives $\alpha_i$, not the line. Recover the weights from stationarity:

$$
\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i
$$

Recover the bias from any support vector. If $\alpha_k > 0$, complementary slackness says the margin is exactly $1$:

$$
y_k(\mathbf{w}\cdot\mathbf{x}_k+b) = 1
$$

Multiply by $y_k$. Because $y_k^2 = 1$,

$$
b = y_k - \mathbf{w}\cdot\mathbf{x}_k
$$

Every support vector must produce the same $b$. If two of them disagree, the multipliers are not optimal.


## The three-point recovery

$$
\mathbf{w}
= \frac{1}{2}(2, 0)
- \frac{1}{4}(0, 1)
- \frac{1}{4}(0, -1)
= (1, 0)
$$

From the positive support vector:

$$
b = (+1) - (1, 0)\cdot(2, 0) = 1 - 2 = -1
$$

From $(0, 1)$:

$$
b = (-1) - (1, 0)\cdot(0, 1) = -1
$$

From $(0, -1)$ the dot product is also $0$, so $b = -1$ again. A new point $(3, 2)$ then scores

$$
(1, 0)\cdot(3, 2) - 1 = 2
$$

and is called positive. Its distance to the line is $|2|/1 = 2$.


In [1]:
# Recover w and b from the multipliers, then score a new point.
from fractions import Fraction

points = [(Fraction(2), Fraction(0)), (Fraction(0), Fraction(1)), (Fraction(0), Fraction(-1))]
labels = [Fraction(1), Fraction(-1), Fraction(-1)]
alpha = [Fraction(1, 2), Fraction(1, 4), Fraction(1, 4)]

w1 = sum(a * y * x1 for a, y, (x1, x2) in zip(alpha, labels, points))
w2 = sum(a * y * x2 for a, y, (x1, x2) in zip(alpha, labels, points))
assert (w1, w2) == (1, 0)

biases = []
for y, (x1, x2) in zip(labels, points):
    biases.append(y - (w1 * x1 + w2 * x2))
assert biases == [Fraction(-1), Fraction(-1), Fraction(-1)]
b = biases[0]

new_point = (Fraction(3), Fraction(2))
new_score = w1 * new_point[0] + w2 * new_point[1] + b
assert new_score == 2
print("recovered w", (w1, w2), "b", b)
print("score at (3, 2)", new_score)


recovered w (Fraction(1, 1), Fraction(0, 1)) b -1
score at (3, 2) 2


## Using a non-support vector to find $b$ is a bug

The formula $b = y_k - \mathbf{w}\cdot\mathbf{x}_k$ needs margin exactly $1$. For $(4, 0)$ with label $+1$, the true score is $3$, so

$$
y - \mathbf{w}\cdot\mathbf{x} = 1 - 4 = -3
$$

which is not the bias. The point is correctly classified and still gives the wrong $b$, because it is not on the edge.


In [2]:
# The bias formula is valid only on support vectors.
from fractions import Fraction

false_bias = Fraction(1) - Fraction(4)  # y - w·x for the point (4, 0)
true_bias = Fraction(-1)
assert false_bias == -3
assert false_bias != true_bias
print("bias from a non-support vector", false_bias)
print("bias from a support vector", true_bias)


bias from a non-support vector -3
bias from a support vector -1


## A pitfall

If several multipliers are positive, average the bias estimates only after you know they agree. Averaging can hide a bad solution: one vector giving $b = -1$ and another giving $b = -3$ should be an error, not a reported bias of $-2$.

## What to carry forward

$\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$ and $b = y_k - \mathbf{w}\cdot\mathbf{x}_k$ for any $k$ with $\alpha_k > 0$. On this example both formulas return $\mathbf{w} = (1, 0)$ and $b = -1$.
